# 13 — Mini-Gemini: The Integration Project

**Welcome to the capstone project.**

In this notebook you will assemble everything from Parts 1–4 into a single, working language model pipeline:

1. Tokenise a real text corpus
2. Configure and build the model
3. Train with the full training recipe
4. Evaluate with perplexity
5. Generate text with multiple strategies
6. Inspect what the model has learned (attention patterns)

**No `# TODO` markers here** — this notebook is a guided integration exercise where you choose your own configurations and explore the results.

---

In [ ]:
import sys, os, math, time
sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

from src.models.transformer import TransformerLM, TransformerConfig
from src.generation.sampling import greedy_decode, sample_decode
from src.utils.visualization import plot_attention_weights

torch.manual_seed(2024)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'PyTorch: {torch.__version__}')

## Step 1 — Choose Your Corpus

Pick one of the options below (or bring your own text).

In [ ]:
# Option A: Built-in fairy tale corpus (always works, no download needed)
CORPUS_OPTION = 'fairy_tales'   # change to 'tinystories' if data is downloaded

if CORPUS_OPTION == 'fairy_tales':
    corpus = """
once upon a time in a kingdom far away there lived a wise king and a kind queen. they had three
children a brave knight a clever princess and a curious young prince. the knight protected the
kingdom from dragons. the princess solved riddles and opened ancient locks. the prince explored
every corner of the great forest and learned the language of birds and beasts.

one day a terrible storm covered the land in darkness. the king called his children together.
he said only the one who brings back the golden lamp from the cave of echoes can lift this curse.
the three children set off together through the forest over the mountain and across the river of stars.

the knight fought the guardian dragon. the princess decoded the ancient inscription on the cave door.
the prince spoke kindly to the river fish who showed them a secret path. together they reached the
cave found the golden lamp and returned the light to the kingdom. the storm faded and the stars
returned and the people celebrated for seven days and seven nights.

from that day the kingdom remembered that courage wisdom and kindness are strongest when they
work together. and they all lived happily ever after.
""".strip().lower() * 8   # repeat to give enough training data

elif CORPUS_OPTION == 'tinystories':
    data_path = '../../data/raw/tinystories_small.txt'
    if os.path.exists(data_path):
        with open(data_path) as f:
            corpus = f.read()
    else:
        print('TinyStories not found. Run: python scripts/download_data.py --dataset tinystories --size small')
        corpus = ''

print(f'Corpus: {len(corpus):,} characters')

## Step 2 — Tokenise

In [ ]:
# Character-level tokeniser (always available)
chars  = sorted(set(corpus))
VOCAB  = len(chars)
c2i    = {c: i for i, c in enumerate(chars)}
i2c    = {i: c for i, c in enumerate(chars)}
encode = lambda s: [c2i.get(c, 0) for c in s]
decode = lambda ids: ''.join(i2c.get(i, '?') for i in ids)

print(f'Vocabulary: {VOCAB} unique characters')
print(f'Characters: {chars}')

# Build training sequences
SEQ_LEN = 64
ids     = torch.tensor(encode(corpus), dtype=torch.long)

# 90/10 train/val split
split   = int(0.9 * len(ids))
train_ids = ids[:split]
val_ids   = ids[split:]

def make_dataset(token_ids, seq_len, stride=None):
    stride = stride or seq_len // 2
    seqs = torch.stack([token_ids[i:i+seq_len] for i in range(0, len(token_ids)-seq_len, stride)])
    return TensorDataset(seqs)

train_ds = make_dataset(train_ids, SEQ_LEN)
val_ds   = make_dataset(val_ids,   SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)

print(f'\nTrain sequences: {len(train_ds)}')
print(f'Val   sequences: {len(val_ds)}')

## Step 3 — Configure & Build the Model

Try different configs and observe the trade-off between model size and training speed.

In [ ]:
# ============================================================
# YOUR CHOICES — experiment with different configurations!
# ============================================================
MODEL_CONFIG = 'small'   # 'tiny', 'small', or 'custom'

configs = {
    'tiny': dict(d_model=64,  n_heads=4, n_layers=2, d_ff=256),
    'small': dict(d_model=128, n_heads=4, n_layers=4, d_ff=512),
}

mc = configs.get(MODEL_CONFIG, configs['small'])

cfg = TransformerConfig(
    vocab_size=VOCAB,
    d_model=mc['d_model'],
    n_heads=mc['n_heads'],
    n_layers=mc['n_layers'],
    d_ff=mc['d_ff'],
    max_seq_len=SEQ_LEN,
    dropout=0.1,
    ffn_type='swiglu',       # try 'standard' or 'swiglu'
    norm_type='rmsnorm',     # try 'layernorm' or 'rmsnorm'
    pos_encoding='sinusoidal', # try 'sinusoidal', 'learned', or 'rope'
    tie_weights=True,
)

model = TransformerLM(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_CONFIG}')
print(f'Parameters: {n_params:,}')
print()
print(cfg)

## Step 4 — Train

In [ ]:
# Training hyperparameters
LEARNING_RATE  = 3e-3
N_EPOCHS       = 30
WARMUP_STEPS   = 50
GRAD_CLIP      = 1.0

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=0.01, betas=(0.9, 0.95)
)

def get_lr(step, warmup, max_steps, peak=LEARNING_RATE, min_lr=1e-5):
    if step < warmup:
        return peak * step / warmup
    progress = (step - warmup) / max(1, max_steps - warmup)
    return min_lr + 0.5 * (peak - min_lr) * (1 + math.cos(math.pi * progress))

total_steps = N_EPOCHS * len(train_loader)
train_losses, val_losses = [], []
step = 0
start = time.time()

for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0.0

    for (batch,) in train_loader:
        batch = batch.to(device)
        x, y = batch[:, :-1], batch[:, 1:]

        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB), y.reshape(-1))

        # LR schedule
        lr = get_lr(step, WARMUP_STEPS, total_steps)
        for g in optimizer.param_groups: g['lr'] = lr

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        epoch_loss += loss.item()
        step += 1

    epoch_loss /= len(train_loader)
    train_losses.append(epoch_loss)

    # Validation
    model.eval()
    with torch.no_grad():
        vl = sum(
            F.cross_entropy(
                model(b[:, :-1].to(device)).view(-1, VOCAB),
                b[:, 1:].reshape(-1).to(device)
            ).item()
            for (b,) in val_loader
        ) / len(val_loader)
    val_losses.append(vl)

    if (epoch+1) % 5 == 0:
        elapsed = time.time() - start
        ppl = math.exp(vl)
        print(f'Epoch {epoch+1:3d}/{N_EPOCHS}  '
              f'train={epoch_loss:.4f}  val={vl:.4f}  '
              f'ppl={ppl:.1f}  lr={lr:.2e}  t={elapsed:.0f}s')

print('\nTraining complete.')

## Step 5 — Evaluate

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs = list(range(1, len(train_losses)+1))
ax1.plot(epochs, train_losses, label='Train', color='steelblue')
ax1.plot(epochs, val_losses,   label='Val',   color='tomato')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, [math.exp(l) for l in train_losses], label='Train PPL', color='steelblue')
ax2.plot(epochs, [math.exp(l) for l in val_losses],   label='Val PPL',   color='tomato')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Perplexity')
ax2.set_title('Perplexity (lower = better)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

final_ppl = math.exp(val_losses[-1])
print(f'Final validation perplexity: {final_ppl:.1f}')
print(f'(Random baseline: {VOCAB:.0f}  |  Perfect: 1.0)')

## Step 6 — Generate Text

In [ ]:
prompts = [
    "once upon a time",
    "the princess",
    "together they",
]

strategies = [
    ('Greedy',         dict(temperature=1.0, top_k=1)),
    ('T=0.8, top-k=10', dict(temperature=0.8, top_k=10, top_p=0.9)),
    ('T=1.2, creative', dict(temperature=1.2, top_k=20, top_p=0.95)),
]

model.eval()
for prompt in prompts:
    p_ids = torch.tensor([encode(prompt)], device=device)
    print(f'\nPrompt: "{prompt}"')
    print('-' * 50)
    for name, kwargs in strategies:
        out = sample_decode(model, p_ids, max_new_tokens=80, eos_token_id=None, **kwargs)
        generated = decode(out[0, len(encode(prompt)):].tolist())
        print(f'{name:<22}: "{generated}"')

## Step 7 — Inspect Attention Patterns

In [ ]:
from src.models.transformer_block import DecoderBlock

probe_text = "the knight fought the dragon"
probe_ids  = torch.tensor([encode(probe_text)], device=device)

model.eval()
with torch.no_grad():
    _ = model(probe_ids)

# Grab attention weights from the first transformer block
chars_in_probe = list(probe_text)

# Try to get attention weights if the block exposes them
try:
    block = model.blocks[0]
    weights = block.attn.get_attention_weights()  # (1, n_heads, seq, seq)
    if weights is not None:
        n_heads = weights.shape[1]
        n_show  = min(n_heads, 4)
        fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))
        if n_show == 1: axes = [axes]
        for h, ax in enumerate(axes):
            w = weights[0, h].cpu().numpy()
            im = ax.imshow(w, cmap='Blues', vmin=0)
            ax.set_xticks(range(len(chars_in_probe)))
            ax.set_xticklabels(chars_in_probe, fontfamily='monospace', fontsize=8)
            ax.set_yticks(range(len(chars_in_probe)))
            ax.set_yticklabels(chars_in_probe, fontfamily='monospace', fontsize=8)
            ax.set_title(f'Layer 1, Head {h+1}')
            plt.colorbar(im, ax=ax)
        plt.suptitle(f'Attention: "{probe_text}"', fontsize=11)
        plt.tight_layout(); plt.show()
    else:
        print('Attention weights not cached — run a forward pass with return_attention=True')
except AttributeError:
    print('Could not access block attention weights directly — model architecture may differ.')

## Step 8 — Reflection

Answer these questions in the cell below (write in the markdown or code cell):

1. What final perplexity did your model achieve? How does it compare to random (= vocab_size)?
2. Did you observe any signs of overfitting? How would you mitigate it?
3. Which generation strategy produced the most coherent text? Why?
4. What patterns do you see in the attention weights? Which characters does each head focus on?
5. What would you change to scale this model to handle a 10x larger dataset?

*(Write your reflections here)*

## Congratulations!

You have:
- Built every component of a transformer from scratch
- Trained a language model end-to-end
- Generated text with multiple decoding strategies
- Inspected what the model learned

**Extension challenges** (see `14_advanced_extensions.ipynb`):
- Train on TinyStories with BPE tokenisation
- Add MoE layers and compare with a dense baseline
- Implement KV-cache for faster generation
- Add image encoding and build a multimodal model